# 04 Modeling: IPEDS Driver Modeling

This notebook explores IPEDS-based drivers connected to earnings differences.


## Notebook workflow

This notebook tests whether program-level IPEDS features help explain residual earnings differences after the main prediction model has already accounted for structural constraints.

The workflow is:
- load the residual outputs and IPEDS driver features,
- check whether the multi-year residual signals agree in direction,
- build a combined residual target,
- compare alternative target definitions,
- test regression models on continuous residual magnitude,
- and test whether the same features work better in a classification setup.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from ira.ingest.ingest_scorecard import save
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL
import math
import json
import time

root = Path.cwd()/'institutional-roi-analysis'
# os.chdir(root/"Seb_branch"/"institutional-roi-analysis"/"notebooks")
pd.set_option("display.max_columns",None)
display(root)

WindowsPath('C:/Users/sebas/PycharmProjects/Git/Seb_branch/institutional-roi-analysis')

In [107]:
residual_df = pd.read_csv(root/"data"/"raw"/"scorecard"/"raw_residual_FL_stable_programs.csv")
ipedsd_df = pd.read_csv(root/"data"/"clean"/"ipeds"/"clean_ipeds_drivers.csv")

In [108]:
residual_df.head()

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score
0,301,3,133492,1,"Private, nonprofit",56.0,27.715798,-82.687043,11,11,0.7580,77695.0,0.411392,2.0,21.0,1,open,10.662914,10.643175,Natural Resources Conservation and Research.,Eckerd College,42741.0,41905.617,835.382813,39932.0,10.594933,50566.960,10.831054,-10634.960938,77,25709.0,10.154596,29107.588,10.278754,-3398.587891,94.0,8,medium,-0.116760,-0.115420,-0.210314,-0.207225,0.019935,0.019527,6.0,8.0,3.0,2.0,-5.0,-3.0,2.516611,0.359516,0.250313,-0.115420
1,301,3,133951,2,Public,36.0,25.757320,-80.373928,21,15,0.5466,23924.0,0.813127,2.0,23.0,1,mid,10.745076,10.783957,Natural Resources Conservation and Research.,Florida International University,46401.0,48240.610,-1839.609375,49748.0,10.814726,55158.484,10.917966,-5410.484375,24,36007.0,10.491469,33721.535,10.425892,2285.464844,21.0,8,medium,0.067775,0.063452,-0.098090,-0.092620,-0.038134,-0.036835,2.0,7.0,6.0,5.0,-1.0,4.0,2.645751,0.377964,0.250313,-0.036835
2,301,3,134097,1,Public,78.0,30.443147,-84.295064,12,15,0.2422,43729.0,0.597532,2.0,20.0,1,elite,10.780705,10.786012,Natural Resources Conservation and Research.,Florida State University,48084.0,48339.850,-255.851562,54288.0,10.902058,52498.200,10.868534,1789.800781,188,30146.0,10.313808,36875.710,10.515308,-6729.710937,161.0,8,medium,-0.182497,-0.181380,0.034093,0.033916,-0.005293,-0.005219,7.0,5.0,5.0,-2.0,0.0,-2.0,1.154701,0.164957,0.250313,-0.005219
3,301,3,134130,1,Public,28.0,29.646290,-82.347911,12,16,0.2420,39127.0,0.667383,2.0,21.0,1,elite,10.895108,10.878085,Natural Resources Conservation and Research.,University of Florida,53912.0,53002.010,909.988281,62683.0,11.045846,57587.316,10.961058,5095.683594,29,34454.0,10.447380,36076.906,10.493408,-1622.906250,29.0,8,medium,-0.044985,-0.042980,0.088486,0.084491,0.017169,0.016391,5.0,3.0,4.0,-2.0,1.0,-1.0,1.000000,0.142857,0.250313,0.016391
4,301,3,136950,1,"Private, nonprofit",29.0,28.592787,-81.349239,21,11,0.4754,43978.0,0.580000,2.0,22.0,1,mid,10.517050,10.832608,Natural Resources Conservation and Research.,Rollins College,36940.0,50645.630,-13705.628906,63363.0,11.056635,52899.145,10.876143,10463.855469,23,22352.0,10.014671,34313.710,10.443300,-11961.710937,18.0,8,medium,-0.348599,-0.322316,0.197808,0.186247,-0.270618,-0.258835,8.0,2.0,8.0,-6.0,6.0,0.0,3.464102,0.494872,0.250313,-0.258835


## Check whether the yearly residual signals are consistent

Before combining the 1-year, 4-year, and 5-year residuals into one target, I first check whether they usually point in the same direction. If they disagree too often, then a combined target could mix signal and noise instead of capturing a stable performance pattern.

In [109]:
# Do the year errors generally agree in direction?
residual_df['sign_agreement'] = (
    np.sign(residual_df['1_year_error']) == 
    np.sign(residual_df['4_year_error'])
) & (
    np.sign(residual_df['4_year_error']) == 
    np.sign(residual_df['5_year_error'])
)

print(residual_df['sign_agreement'].value_counts(normalize=True))

# Also check correlation between year errors
print(residual_df[['1_year_error','4_year_error','5_year_error']].corr())

sign_agreement
False    0.572056
True     0.427944
Name: proportion, dtype: float64
              1_year_error  4_year_error  5_year_error
1_year_error      1.000000      0.593404      0.407938
4_year_error      0.593404      1.000000      0.524356
5_year_error      0.407938      0.524356      1.000000


## Build the combined residual target

After checking year-to-year agreement, I create a combined residual target and a smooth sample-size weight. The target summarizes unexplained earnings performance across the available windows, and the weight reduces the influence of very small programs.

In [110]:
residual_df['total_pred'] = residual_df[
    ["1_year_pred", "4_year_pred", "5_year_pred"]
].sum(axis=1)

residual_df["total_count"] = (
    residual_df["1_yr_working_count"] +
    residual_df["4_yr_working_count"] +
    residual_df["5_yr_working_count"]
)

k = np.percentile(np.log1p(residual_df["total_count"]), 75)

residual_df["weight"] = (
    np.log1p(residual_df["total_count"]) /
    np.log1p(residual_df["total_count"] + k)
)

residual_df["combined_pct_error"] = residual_df[
    ["1_year_error", "4_year_error", "5_year_error"]
].median(axis=1) / residual_df['total_pred'] * 3

# consistent_mask = residual_df['sign_agreement'] == True
# residual_df = residual_df[consistent_mask]
# print(f"Consistent schools: {consistent_mask.sum()} of {len(residual_df)}")

residual_df

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score,sign_agreement,total_pred,total_count,weight,combined_pct_error
0,301,3,133492,1,"Private, nonprofit",56.0,27.715798,-82.687043,11,11,0.7580,77695.0,0.411392,2.0,21.0,1,open,10.662914,10.643175,Natural Resources Conservation and Research.,Eckerd College,42741.0,41905.617,835.382813,39932.0,10.594933,50566.960,10.831054,-10634.960938,77,25709.0,10.154596,29107.588,10.278754,-3398.587891,94.0,8,medium,-0.116760,-0.115420,-0.210314,-0.207225,0.019935,0.019527,6.0,8.0,3.0,2.0,-5.0,-3.0,2.516611,0.359516,0.250313,-0.115420,False,121580.165,227.0,0.995038,-0.083860
1,301,3,133951,2,Public,36.0,25.757320,-80.373928,21,15,0.5466,23924.0,0.813127,2.0,23.0,1,mid,10.745076,10.783957,Natural Resources Conservation and Research.,Florida International University,46401.0,48240.610,-1839.609375,49748.0,10.814726,55158.484,10.917966,-5410.484375,24,36007.0,10.491469,33721.535,10.425892,2285.464844,21.0,8,medium,0.067775,0.063452,-0.098090,-0.092620,-0.038134,-0.036835,2.0,7.0,6.0,5.0,-1.0,4.0,2.645751,0.377964,0.250313,-0.036835,False,137120.629,81.0,0.983585,-0.040248
2,301,3,134097,1,Public,78.0,30.443147,-84.295064,12,15,0.2422,43729.0,0.597532,2.0,20.0,1,elite,10.780705,10.786012,Natural Resources Conservation and Research.,Florida State University,48084.0,48339.850,-255.851562,54288.0,10.902058,52498.200,10.868534,1789.800781,188,30146.0,10.313808,36875.710,10.515308,-6729.710937,161.0,8,medium,-0.182497,-0.181380,0.034093,0.033916,-0.005293,-0.005219,7.0,5.0,5.0,-2.0,0.0,-2.0,1.154701,0.164957,0.250313,-0.005219,False,137713.760,427.0,0.997610,-0.005574
3,301,3,134130,1,Public,28.0,29.646290,-82.347911,12,16,0.2420,39127.0,0.667383,2.0,21.0,1,elite,10.895108,10.878085,Natural Resources Conservation and Research.,University of Florida,53912.0,53002.010,909.988281,62683.0,11.045846,57587.316,10.961058,5095.683594,29,34454.0,10.447380,36076.906,10.493408,-1622.906250,29.0,8,medium,-0.044985,-0.042980,0.088486,0.084491,0.017169,0.016391,5.0,3.0,4.0,-2.0,1.0,-1.0,1.000000,0.142857,0.250313,0.016391,False,146666.232,86.0,0.984685,0.018613
4,301,3,136950,1,"Private, nonprofit",29.0,28.592787,-81.349239,21,11,0.4754,43978.0,0.580000,2.0,22.0,1,mid,10.517050,10.832608,Natural Resources Conservation and Research.,Rollins College,36940.0,50645.630,-13705.628906,63363.0,11.056635,52899.145,10.876143,10463.855469,23,22352.0,10.014671,34313.710,10.443300,-11961.710937,18.0,8,medium,-0.348599,-0.322316,0.197808,0.186247,-0.270618,-0.258835,8.0,2.0,8.0,-6.0,6.0,0.0,3.464102,0.494872,0.250313,-0.258835,False,137858.485,70.0,0.980570,-0.260304
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1362,5401,3,134097,1,Public,119.0,30.443147,-84.295064,12,15,0.2422,43729.0,0.597532,2.0,20.0,1,elite,10.814444,10.873309,History.,Florida State University,49734.0,52749.477,-3015.476563,56834.0,10.947890,54684.008,10.909327,2149.992187,82,27944.0,10.237958,31843.193,10.368579,-3899.193359,82.0,8,medium,-0.122450,-0.120802,0.039317,0.038780,-0.057166,-0.056680,5.0,3.0,6.0,-2.0,3.0,1.0,1.527525,0.218218,0.339303,-0.056

In [111]:
ipedsd_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6909 entries, 0 to 6908
Data columns (total 31 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   unit_id                                   6909 non-null   int64  
 1   year                                      6909 non-null   int64  
 2   code                                      6909 non-null   int64  
 3   credential_level                          6909 non-null   int64  
 4   program_completers                        6909 non-null   int64  
 5   program_completers_log                    6909 non-null   float64
 6   program_completer_share_within_school     6906 non-null   float64
 7   instruction_salary_pct                    0 non-null      float64
 8   academic_support_salary_pct               0 non-null      float64
 9   student_services_salary_pct               0 non-null      float64
 10  research_salary_pct                 

In [112]:
ipedsd_df=ipedsd_df.drop(columns=['instruction_salary_pct', 'academic_support_salary_pct',
       'student_services_salary_pct', 'research_salary_pct'])

In [113]:
ipedsd_df.head()

,unit_id,year,code,credential_level,program_completers,program_completers_log,program_completer_share_within_school,no_ap_credit,study_abroad,career_counseling,employment_services,placement_services,instruction_expense_pct,research_expense_pct,student_service_expense_pct,endowment_per_fte,equity_ratio,staff_per_student,instructional_fte_per_student,irps_fte_per_student,instructional_share_of_staff,research_share_of_staff,instructional_staff_total,instructional_staff_short_contract,instructional_staff_long_contract,instructional_staff_long_contract_share,instructional_staff_short_contract_share
0,132374,2020,1101,1,14,2.708050,0.016588,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN
1,132374,2020,1102,1,10,2.397895,0.011848,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN
2,132374,2020,1108,1,40,3.713572,0.047393,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN
3,132374,2020,1109,1,34,3.555348,0.040284,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN
4,132374,2020,1205,1,47,3.871201,0.055687,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN


## Merge residual targets with IPEDS drivers

Once the residual target and IPEDS driver data are ready, I merge them using institution, program, and credential identifiers. This creates the program-level explanatory dataset used in the rest of the notebook.

In [114]:
targ='combined_pct_error'
merge_df=ipedsd_df.merge(residual_df[["unit_id","code","credential_level","school_name",targ,'weight']], on = ["unit_id","code","credential_level"], how="right")

In [115]:
model_df=merge_df.copy()

print("ipedsd_df shape:", ipedsd_df.shape)
print("residual_df shape:", residual_df.shape)
print("model_df shape:", model_df.shape)

print("\nmodel_df columns:")
print(sorted(model_df.columns.tolist()))

ipedsd_df shape: (6909, 27)
residual_df shape: (1367, 59)
model_df shape: (1367, 30)

model_df columns:
['career_counseling', 'code', 'combined_pct_error', 'credential_level', 'employment_services', 'endowment_per_fte', 'equity_ratio', 'instruction_expense_pct', 'instructional_fte_per_student', 'instructional_share_of_staff', 'instructional_staff_long_contract', 'instructional_staff_long_contract_share', 'instructional_staff_short_contract', 'instructional_staff_short_contract_share', 'instructional_staff_total', 'irps_fte_per_student', 'no_ap_credit', 'placement_services', 'program_completer_share_within_school', 'program_completers', 'program_completers_log', 'research_expense_pct', 'research_share_of_staff', 'school_name', 'staff_per_student', 'student_service_expense_pct', 'study_abroad', 'unit_id', 'weight', 'year']


In [116]:
model_df=model_df.drop(columns=[
    'year',
    # 'program_completer_share_within_school','program_completers',
    # 'program_completers_log',"credential_level",
    'code',
    # 'unit_id',
    # 'school_name'
    ]
)

In [117]:
inst_model_df=model_df.copy()

## Inspect the merged modeling dataset

Before fitting models, I check the merged dataset shape, review the available features, and look at simple correlations with the target. This gives a first read on whether any IPEDS variables appear meaningfully related to residual performance.

In [118]:
display(inst_model_df.describe())
display("Feature correlation to target variable",inst_model_df.corr(numeric_only=True)[targ].sort_values())
display(inst_model_df.info())

# drop_cols=[
#     "instruction_expense_pct",
#     "student_service_expense_pct",
#     "career_counseling",
#     "employment_services",
# ]

,unit_id,credential_level,program_completers,program_completers_log,program_completer_share_within_school,no_ap_credit,study_abroad,career_counseling,employment_services,placement_services,instruction_expense_pct,research_expense_pct,student_service_expense_pct,endowment_per_fte,equity_ratio,staff_per_student,instructional_fte_per_student,irps_fte_per_student,instructional_share_of_staff,research_share_of_staff,instructional_staff_total,instructional_staff_short_contract,instructional_staff_long_contract,instructional_staff_long_contract_share,instructional_staff_short_contract_share,combined_pct_error,weight
count,1367.000000,1367.000000,1316.000000,1316.000000,1315.000000,1316.000000,1316.000000,1316.000000,1316.000000,1316.000000,748.000000,748.000000,748.000000,686.000000,704.000000,1316.000000,1316.000000,1316.000000,1316.000000,1316.000000,1316.000000,1316.000000,1316.000000,1183.000000,1183.000000,1367.000000,1367.000000
mean,209880.422824,2.917337,165.954407,4.250327,0.100120,0.101064,0.703647,0.985562,0.940729,0.943009,36.225936,9.971925,7.387701,10177.778426,56.313920,0.110172,0.034979,0.039955,0.370487,0.014366,1248.325228,2.525836,1245.799392,0.995769,0.004231,0.008198,0.993004
std,133270.881787,1.334216,460.199530,1.275426,0.210793,0.301528,0.456822,0.119332,0.236220,0.231913,8.745151,9.432236,4.028275,10376.981780,14.719938,0.095839,0.020350,0.027559,0.116590,0.020912,1406.773408,21.728713,1406.775843,0.027108,0.027108,0.124116,0.006231
min,132471.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,26.000000,0.000000,0.000000,227.000000,37.000000,0.016738,0.007823,0.007823,0.131088,0.000000,0.000000,0.000000,0.000000,0.708333,0.000000,-0.510359,0.972659
25%,133951.000000,2.000000,32.000000,3.496508,0.007541,0.000000,0.000000,1.000000,1.000000,1.000000,31.000000,0.000000,5.000000,2659.000000,43.000000,0.053376,0.021183,0.021183,0.292579,0.000000,152.500000,0.000000,152.500000,1.000000,0.000000,-0.061837,0.989595
50%,136215.000000,3.000000,67.500000,4.226807,0.021624,0.000000,1.000000,1.000000,1.000000,1.000000,36.000000,10.000000,7.000000,6222.000000,57.000000,0.083549,0.032074,0.032123,0.356119,0.000000,678.000000,0.000000,678.000000,1.000000,0.000000,-0.000934,0.994987
75%,138354.000000,3.000000,147.250000,4.998896,0.063495,0.000000,1.000000,1.000000,1.000000,1.000000,38.000000,17.000000,10.000000,12079.000000,69.000000,0.143447,0.046654,0.046654,0.435897,0.033449,2494.000000,0.000000,2494.000000,1.000000,0.000000,0.066193,0.998099
max,499495.000000,7.000000,9082.000000,9.114160,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,73.000000,25.000000,18.000000,55596.000000,89.000000,0.937500,0.375000,0.375000,0.723684,0.063864,5044.000000,238.000000,5044.000000,1.000000,0.291667,0.947358,0.999975


'Feature correlation to target variable'

research_expense_pct                       -0.035123
staff_per_student                          -0.031073
equity_ratio                               -0.029533
instructional_staff_long_contract_share    -0.027951
placement_services                         -0.019911
irps_fte_per_student                       -0.018555
research_share_of_staff                    -0.017926
program_completers                         -0.016503
unit_id                                    -0.014421
instructional_fte_per_student              -0.011990
career_counseling                          -0.009805
study_abroad                               -0.009214
endowment_per_fte                          -0.007609
weight                                     -0.006370
credential_level                           -0.003820
no_ap_credit                               -0.002353
instructional_staff_long_contract          -0.002132
instructional_staff_total                  -0.001851
program_completers_log                      0.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1367 entries, 0 to 1366
Data columns (total 28 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   unit_id                                   1367 non-null   int64  
 1   credential_level                          1367 non-null   int64  
 2   program_completers                        1316 non-null   float64
 3   program_completers_log                    1316 non-null   float64
 4   program_completer_share_within_school     1315 non-null   float64
 5   no_ap_credit                              1316 non-null   float64
 6   study_abroad                              1316 non-null   float64
 7   career_counseling                         1316 non-null   float64
 8   employment_services                       1316 non-null   float64
 9   placement_services                        1316 non-null   float64
 10  instruction_expense_pct             

None

In [119]:
import plotly.express as px
px.histogram(inst_model_df[targ])

In [120]:
len(inst_model_df[targ])

1367

In [121]:
inst_model_df[targ].skew()

np.float64(0.8030504621696949)

In [122]:
inst_model_df[targ].describe()

count    1367.000000
mean        0.008198
std         0.124116
min        -0.510359
25%        -0.061837
50%        -0.000934
75%         0.066193
max         0.947358
Name: combined_pct_error, dtype: float64

## Compare alternative target definitions

Residual modeling can depend a lot on how the target is defined. This section creates several reasonable target variants so I can test whether the conclusions are tied to one specific construction or whether the same general pattern holds across alternatives.

In [123]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer

def build_target_variants(df):
    d = df.copy()

    e1 = d["1_year_error"]
    e4 = d["4_year_error"]
    e5 = d["5_year_error"]

    c1 = d["1_yr_working_count"]
    c4 = d["4_yr_working_count"]
    c5 = d["5_yr_working_count"]

    total_count = d["total_count"].replace(0, np.nan)
    total_pred  = d["total_pred"].replace(0, np.nan)

    targets = {}

    targets["pct_sum_over_pred"] = (e1 + e4 + e5) / total_pred
    targets["pct_mean_over_pred"] = (e1 + e4 + e5) / (3 * total_pred)
    targets["pct_count_weighted"] = (e1*c1 + e4*c4 + e5*c5) / (total_count * total_pred)

    log_e1 = d["1_year_earning_log"] - d["1_year_pred_log"]
    log_e4 = d["4_year_earning_log"] - d["4_year_pred_log"]
    log_e5 = d["5_year_earning_log"] - d["5_year_pred_log"]

    targets["log_error_sum"] = log_e1 + log_e4 + log_e5
    targets["log_error_mean"] = targets["log_error_sum"] / 3
    targets["log_error_count_weighted"] = (log_e1*c1 + log_e4*c4 + log_e5*c5) / total_count

    sign_mask = (
        (np.sign(e1) == np.sign(e4)) & (np.sign(e4) == np.sign(e5))
    )
    targets["pct_sum_sign_consistent"] = targets["pct_sum_over_pred"].where(sign_mask)

    return targets, sign_mask

def run_target_screen(merge_df, feature_cols, target_variants, sign_mask=None, weight_col="weight"):
    results = []

    for name, target_series in target_variants.items():
        df_tmp = merge_df[feature_cols + [weight_col]].copy()
        df_tmp["__target__"] = target_series.reindex(merge_df.index)

        if sign_mask is not None and name == "pct_sum_sign_consistent":
            df_tmp.loc[~sign_mask.reindex(merge_df.index).fillna(False), "__target__"] = np.nan

        df_tmp = df_tmp.dropna(subset=["__target__"])

        X = df_tmp[feature_cols].apply(pd.to_numeric, errors="coerce")
        X = X.loc[:, X.notna().any()]
        X = X.loc[:, X.nunique(dropna=True) > 1]

        if X.shape[1] == 0:
            print(f"Skipping {name} — no usable features")
            continue

        keep_idx = X.index
        y = df_tmp.loc[keep_idx, "__target__"]
        w = df_tmp.loc[keep_idx, weight_col]

        if len(y) < 50:
            print(f"Skipping {name} — too few rows ({len(y)})")
            continue

        lo, hi = y.quantile(0.02), y.quantile(0.98)
        y = y.clip(lo, hi)

        corr = X.corrwith(y).abs().dropna().sort_values(ascending=False)
        top_corr = corr.iloc[0] if len(corr) else np.nan
        top_feat = corr.index[0] if len(corr) else None
        mean_corr = corr.mean() if len(corr) else np.nan

        imputer = SimpleImputer(strategy="median")
        X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

        fold_scores = []
        for train_idx, test_idx in kf.split(X_imp):
            X_train, X_test = X_imp.iloc[train_idx], X_imp.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            w_train, w_test = w.iloc[train_idx], w.iloc[test_idx]

            rf.fit(X_train, y_train, sample_weight=w_train)
            preds = rf.predict(X_test)
            fold_scores.append(r2_score(y_test, preds, sample_weight=w_test))

        rf_r2 = np.mean(fold_scores)

        results.append({
            "target": name,
            "n_rows": len(y),
            "n_features": X.shape[1],
            "top_corr_feat": top_feat,
            "top_corr": round(top_corr, 4) if pd.notna(top_corr) else np.nan,
            "mean_abs_corr": round(mean_corr, 4) if pd.notna(mean_corr) else np.nan,
            "rf_cv_r2": round(rf_r2, 4) if pd.notna(rf_r2) else np.nan,
        })

        print(f"✓ {name:30s} | n={len(y):4d} | p={X.shape[1]:3d} | top_corr={top_corr:.3f} ({top_feat}) | RF R²={rf_r2:.3f}")

    return pd.DataFrame(results).sort_values("rf_cv_r2", ascending=False)

# ── RUN IT ───────────────────────────────────────────────────────
target_variants, sign_mask = build_target_variants(residual_df)

feature_cols = [c for c in merge_df.columns if c in inst_model_df.columns 
                and c not in ["unit_id", "code", "credential_level", 
                               "school_name", "weight", "avg_rank_stability",'combined_pct_error']]

results_df = run_target_screen(merge_df, feature_cols, target_variants, sign_mask)
print("\n", results_df)

✓ pct_sum_over_pred              | n=1367 | p= 23 | top_corr=0.062 (instruction_expense_pct) | RF R²=-0.229
✓ pct_mean_over_pred             | n=1367 | p= 23 | top_corr=0.062 (instruction_expense_pct) | RF R²=-0.228
✓ pct_count_weighted             | n=1367 | p= 23 | top_corr=0.056 (instruction_expense_pct) | RF R²=-0.220
✓ log_error_sum                  | n=1367 | p= 23 | top_corr=0.068 (instruction_expense_pct) | RF R²=-0.227
✓ log_error_mean                 | n=1367 | p= 23 | top_corr=0.068 (instruction_expense_pct) | RF R²=-0.226
✓ log_error_count_weighted       | n=1367 | p= 23 | top_corr=0.059 (instruction_expense_pct) | RF R²=-0.227
✓ pct_sum_sign_consistent        | n= 585 | p= 23 | top_corr=0.050 (instruction_expense_pct) | RF R²=-0.218

                      target  n_rows  n_features            top_corr_feat  \
6   pct_sum_sign_consistent     585          23  instruction_expense_pct   
2        pct_count_weighted    1367          23  instruction_expense_pct   
4            l

## Target-screening takeaway

The screening step suggests that some target definitions are more learnable than others, but none look strong. That matters because it suggests the challenge is not just target choice. It also reflects limited explanatory signal in the available IPEDS features.

## Regression setup: explain continuous residual magnitude

The first modeling strategy treats this as a regression problem. I use the log-transformed absolute residual target so the model focuses on the size of the unexplained difference, regardless of direction.

This asks whether the program-level IPEDS driver set can explain how large the residual earnings gap tends to be.

In [124]:
from sklearn.model_selection import GridSearchCV, KFold, train_test_split, StratifiedKFold
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score

# Reproducibility
RANDOM_STATE = 42

In [125]:
# -------------------
# 1. Define target
# -------------------

# Drop target and target-like columns from X
feature_cols = [c for c in inst_model_df.columns if c not in ["unit_id", targ, "weight",'credential_level','code','avg_rank_stability','school_name']]

X = inst_model_df[feature_cols]

y = np.log1p(inst_model_df[targ].abs())
# y = y.clip(upper=y.quantile(0.9))

w = inst_model_df["weight"]

# Force predictors numeric
X = X.apply(pd.to_numeric, errors="coerce")

# -------------------
# 2. Train/test split
# -------------------
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.2, random_state=RANDOM_STATE
)

# -------------------
# 4. Recompute numeric columns
# -------------------
num_cols = X_train.columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols)
])

In [126]:
len(y)

1367

In [127]:
X.columns

Index(['program_completers', 'program_completers_log',
       'program_completer_share_within_school', 'no_ap_credit', 'study_abroad',
       'career_counseling', 'employment_services', 'placement_services',
       'instruction_expense_pct', 'research_expense_pct',
       'student_service_expense_pct', 'endowment_per_fte', 'equity_ratio',
       'staff_per_student', 'instructional_fte_per_student',
       'irps_fte_per_student', 'instructional_share_of_staff',
       'research_share_of_staff', 'instructional_staff_total',
       'instructional_staff_short_contract',
       'instructional_staff_long_contract',
       'instructional_staff_long_contract_share',
       'instructional_staff_short_contract_share'],
      dtype='object')

In [128]:
# -------------------
# 5. Ridge model
# -------------------
ridge_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", Ridge())
])

# ridge_model = TransformedTargetRegressor(
#     regressor=ridge_pipe,
#     transformer=PowerTransformer(method="yeo-johnson", standardize=False)
# )

ridge_param_grid = {
    "reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0, 100.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

ridge_grid = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=ridge_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

ridge_grid.fit(
    X_train,
    y_train,
    reg__sample_weight=w_train
)

ridge_best = ridge_grid.best_estimator_
ridge_preds = ridge_best.predict(X_test)

print("RIDGE")
print("Best params:", ridge_grid.best_params_)
print("Best CV MAE:", round(-ridge_grid.best_score_, 4))
print("Test MAE (vs unclipped y_test):", round(mean_absolute_error(y_test, ridge_preds), 4))
print("Test R2 (vs unclipped y_test):", round(r2_score(y_test, ridge_preds), 4))

Fitting 5 folds for each of 7 candidates, totalling 35 fits
RIDGE
Best params: {'reg__alpha': 100.0}
Best CV MAE: 0.056
Test MAE (vs unclipped y_test): 0.0549
Test R2 (vs unclipped y_test): -0.08


In [129]:
enet_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", ElasticNet(max_iter=100000))
])

# enet_model = TransformedTargetRegressor(
#     regressor=enet_pipe,
#     transformer=PowerTransformer(method="yeo-johnson", standardize=False)
# )

enet_param_grid = {
    "reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0],
    "reg__l1_ratio": [0.005,0.1, 0.3, 0.5, 0.7, 0.9]
}

enet_grid = GridSearchCV(
    estimator=enet_pipe,
    param_grid=enet_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

enet_grid.fit(
    X_train, 
    y_train,
    reg__sample_weight=w_train
)

enet_best = enet_grid.best_estimator_
enet_preds = enet_best.predict(X_test)

print("\nELASTIC NET")
print("Best params:", enet_grid.best_params_)
print("Best CV MAE:", round(-enet_grid.best_score_, 4))
print("Test MAE (vs unclipped y_test):", round(mean_absolute_error(y_test, enet_preds), 4))
print("Test R2 (vs unclipped y_test):", round(r2_score(y_test, enet_preds), 4))

Fitting 5 folds for each of 36 candidates, totalling 180 fits

ELASTIC NET
Best params: {'reg__alpha': 0.01, 'reg__l1_ratio': 0.3}
Best CV MAE: 0.0558
Test MAE (vs unclipped y_test): 0.0545
Test R2 (vs unclipped y_test): -0.0376


## Linear regression results summary

The linear models provide no explanatory power. Even after regularization and weighting, the fit remains limited, which suggests that the IPEDS feature set does not explain continuous residual magnitude in this setup.

In [40]:
y_pred = [y.mean()] * len(y)

mean_absolute_error(y, y_pred)

0.06301508417497333

## Nonlinear regression follow-up

Because the relationship between IPEDS variables and residual performance may be nonlinear, I also test boosting-based models. This checks whether interactions and nonlinear structure improve on the regularized linear baseline.

In [130]:
from sklearn.ensemble import HistGradientBoostingRegressor

def run_hgbr():
    hgbr_model = Pipeline([
        ('preprocessor',preprocessor),
        ('model',HistGradientBoostingRegressor( 
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        random_state=42 
        ))
    ])
    

    hgbr_model.fit(
        X_train, 
        y_train,
        model__sample_weight=w_train
    ) 
    preds = hgbr_model.predict(X_test) 
    print("MAE:", mean_absolute_error(y_test, preds)) 
    print("R2:", r2_score(y_test, preds))
run_hgbr()

MAE: 0.05455827662714327
R2: -0.07469074623111882


In [131]:
def run_hgbr_2():
    hgbr_model_2 = Pipeline([
        ('preprocessor',preprocessor),
        ('model',HistGradientBoostingRegressor(
        max_depth=5,              # allow more interactions
        learning_rate=0.01,       # slower learning
        max_iter=750,             # more trees
        min_samples_leaf=10,      # regularization
        l2_regularization=2.0,    # stabilize
        random_state=42
        ))
    ])
  

    hgbr_model_2.fit(
        X_train, 
        y_train,
        model__sample_weight=w_train
    ) 

    preds_2 = hgbr_model_2.predict(X_test) 
    print("MAE:", mean_absolute_error(y_test, preds_2)) 
    print("R2:", r2_score(y_test, preds_2))
run_hgbr_2()

MAE: 0.05607041930186623
R2: -0.1293154921516868


In [132]:
from sklearn.model_selection import cross_val_score

# bin target into quantiles
y_bins = pd.qcut(y, q=5, labels=False, duplicates="drop")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_hgbr_model = Pipeline([
        ('preprocessor',preprocessor),
        ('model',HistGradientBoostingRegressor( 
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        random_state=42 
        ))
    ])

y_bins = pd.qcut(y, q=5, labels=False, duplicates="drop")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for train_idx, test_idx in cv.split(X, y_bins):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    w_train, w_test = w.iloc[train_idx], w.iloc[test_idx]

    cv_hgbr_model.fit(X_train, y_train, model__sample_weight=w_train)
    preds = cv_hgbr_model.predict(X_test)

    score = r2_score(y_test, preds, sample_weight=w_test)
    scores.append(score)

print("HistGradientBoostingRegressor")
print(scores)
print("mean:", np.mean(scores))
print("std:", np.std(scores))

HistGradientBoostingRegressor
[0.029912704439342797, 0.013827815557112544, -0.011395511256790503, 0.022292438195004927, 0.0771076529333522]
mean: 0.026349019973604393
std: 0.028937949894496804


In [133]:
y_shuffled = y.sample(frac=1, random_state=42).reset_index(drop=True)

cv_hgbr_model.fit(X_train, y_shuffled.loc[X_train.index], model__sample_weight=w_train)
preds = cv_hgbr_model.predict(X_test)

print("R2 shuffled:", r2_score(y_test, preds))

R2 shuffled: -0.02392390013513368


In [134]:
# bin target into quantiles
scores = []

for train_idx, test_idx in cv.split(X, y_bins):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    w_train, w_test = w.iloc[train_idx], w.iloc[test_idx]

    ridge_best.fit(X_train, y_train, reg__sample_weight=w_train)
    preds = ridge_best.predict(X_test)

    score = r2_score(y_test, preds, sample_weight=w_test)
    scores.append(score)

print("HistGradientBoostingRegressor")
print(scores)
print("mean:", np.mean(scores))
print("std:", np.std(scores))

HistGradientBoostingRegressor
[0.02996391467049364, -0.006794774776300194, -0.018404091395213262, 0.016461881903720443, 0.018945817618049032]
mean: 0.008034549604149932
std: 0.01783190035327317


In [135]:
from xgboost import XGBRegressor

cv_xgb_model = Pipeline([
        ('preprocessor',preprocessor),
        ('model',XGBRegressor(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=42
        ))
    ])

scores = []

for train_idx, test_idx in cv.split(X, y_bins):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    w_train, w_test = w.iloc[train_idx], w.iloc[test_idx]

    cv_xgb_model.fit(X_train, y_train, model__sample_weight=w_train)
    preds = cv_xgb_model.predict(X_test)

    score = r2_score(y_test, preds, sample_weight=w_test)
    scores.append(score)

print("HistGradientBoostingRegressor")
print(scores)
print("mean:", np.mean(scores))
print("std:", np.std(scores))

HistGradientBoostingRegressor
[-0.003879319485449173, -0.008987834670408423, -0.07200936366722877, 0.005942015793285349, 0.03820607086845629]
mean: -0.008145686232268945
std: 0.03590105605525349


## Nonlinear regression takeaway

The nonlinear models perform somewhat better than the linear ones, but the gains are still urrelavent. This suggests that the IPEDS features contain some signal, but not enough to make continuous residual prediction.

In [136]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score



# -------------------
# 1. Target (top vs bottom)
# -------------------
y_raw = inst_model_df[targ].abs()
# y_raw = y.clip(upper=y.quantile(0.95))
# y_raw = inst_model_df[targ].copy()

low = y_raw.quantile(0.25)
high = y_raw.quantile(0.75)

mask = (y_raw <= low) | (y_raw >= high)

# -------------------
# 2. Use ALL columns except target
# -------------------
X_clf = inst_model_df.drop(columns=["unit_id", targ, "weight",'credential_level','code','avg_rank_stability','school_name'], errors="ignore").loc[mask].copy()
y_clf = (y_raw.loc[mask] >= high).astype(int)
w_clf = inst_model_df['weight'].loc[mask].copy()

# force everything numeric
X_clf = X_clf.apply(pd.to_numeric, errors="coerce")

# -------------------
# 3. Train/test split
# -------------------
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_clf,
    y_clf,
    w_clf,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y_clf
)

# -------------------
# 4. Preprocessing (CRITICAL)
# -------------------
num_cols = X_train.columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols)
])

## Reframe the problem as tail classification

Since continuous regression is difficult, I also test whether the IPEDS features work better for separating cases in the tails of the residual distribution. This turns the problem into a binary classification task using the lower and upper parts of the target.

This is an easier problem than full regression, so the classification results should be interpreted separately.

In [137]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# -------------------
# 5. Model
# -------------------
clf = Pipeline([
    ("preprocessor", preprocessor),
    ("logit", LogisticRegression(max_iter=10000))
])

# -------------------
# 6. Fit
# -------------------
clf.fit(
    X_train, 
    y_train,
    logit__sample_weight=w_train
)

# -------------------
# 7. Evaluate
# -------------------
preds = clf.predict(X_test)
probs = clf.predict_proba(X_test)[:, 1]

print(y_clf.value_counts())
print("Accuracy:", round(accuracy_score(y_test, preds), 4))
print("ROC AUC:", round(roc_auc_score(y_test, probs), 4))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, preds))
print("\nClassification Report:\n", classification_report(y_test, preds))

combined_pct_error
0    342
1    342
Name: count, dtype: int64
Accuracy: 0.5874
ROC AUC: 0.6036

Confusion Matrix:
 [[59 44]
 [41 62]]

Classification Report:
               precision    recall  f1-score   support

           0       0.59      0.57      0.58       103
           1       0.58      0.60      0.59       103

    accuracy                           0.59       206
   macro avg       0.59      0.59      0.59       206
weighted avg       0.59      0.59      0.59       206



## Logistic classification baseline

This section provides a simple linear baseline for classifying lower- versus higher-residual cases. It answers whether the features contain enough signal to separate the two groups even before using a more flexible nonlinear classifier.

In [57]:
from sklearn.ensemble import HistGradientBoostingClassifier

clf_hgb = Pipeline([
    ("preprocessor", preprocessor),
    ("logit", HistGradientBoostingClassifier(
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        random_state=42
    ))
])


clf_hgb.fit(
    X_train, 
    y_train,
    logit__sample_weight=w_train)

preds_hgb = clf_hgb.predict(X_test)
probs_hgb = clf_hgb.predict_proba(X_test)[:, 1]

print(y_clf.value_counts())
print("HGB Accuracy:", round(accuracy_score(y_test, preds_hgb), 4))
print("HGB ROC AUC:", round(roc_auc_score(y_test, probs_hgb), 4))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, preds_hgb))
print("\nClassification Report:\n", classification_report(y_test, preds_hgb))

combined_pct_error
1    147
0    147
Name: count, dtype: int64
HGB Accuracy: 0.6966
HGB ROC AUC: 0.7341

Confusion Matrix:
 [[33 12]
 [15 29]]

Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.73      0.71        45
           1       0.71      0.66      0.68        44

    accuracy                           0.70        89
   macro avg       0.70      0.70      0.70        89
weighted avg       0.70      0.70      0.70        89



## Classification takeaway

The nonlinear classifier performs noticeably better than the logistic baseline, which suggests that the useful signal in these IPEDS features is at least partly nonlinear. Even so, this result should still be interpreted as tail-separation performance, not proof that the features strongly explain the full continuous residual process.

## Classification takeaway

The nonlinear classifier performs noticeably better than the logistic baseline, which suggests that the useful signal in these IPEDS features is at least partly nonlinear. Even so, this result should still be interpreted as tail-separation performance, not proof that the features strongly explain the full continuous residual process.

# Final takeaway

In this notebook, program-level IPEDS features showed limited to moderate value for explaining continuous residual earnings differences. Alternative target definitions and nonlinear models improved the story somewhat, but continuous regression remained challenging.

The strongest results appeared in the tail-classification setup, especially with the nonlinear classifier. That suggests these features may be more useful for separating unusually high- and low-residual cases than for precisely modeling the size of the residual gap.